# Week 3 Day 5 - Lab 2: The final teardown

## Deleting everything and checking what's left

Lab 1 deployed all of UnderCover to AWS for the handover. This is the end of the course, so in this lab we delete all of it, including the app and the context layer on Lambda, which have stayed up since Days 2 and 3, and the CDK bootstrap. Then we check three ways that nothing is left: UnderCover's resources, service by service; everything on your AWS account in us-east-1, from AWS Resource Explorer; and what your account has spent.

## 6 parts:

PART 1: What's there now

PART 2: Both runtimes

PART 3: The Gateway, the Policy and both Memories

PART 4: The app and the context layer

PART 5: The CDK bootstrap and the token vault

PART 6: The three checks

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">You need Lab 1's setup, with the AgentCore CLI at 1.0.0-rc.5, and a terminal open in the <code>undercover</code> folder for the CLI's commands. If you stopped partway through Lab 1, this lab still works: each part finds what to delete by name, and skips what isn't there.<br/><br/>This lab costs nothing: deleting is free, and so are Resource Explorer and reading your budgets. What it deletes is gone for good. Your Supabase database keeps the claims, the decisions and the Copilot's conversations.<br/><br/>On the free route there's nothing on AWS to delete. The last step of <a href="week3day5_lab1free.ipynb">week3day5_lab1free.ipynb</a> covers your Render services.
            </span>
        </td>
    </tr>
</table>

In [ ]:
import time
from collections import Counter
from datetime import UTC, datetime, timedelta
from pathlib import Path

import boto3
from dotenv import dotenv_values, unset_key

import week3day4_aws as aws

## Connect to AWS

boto3 uses the access key that `aws configure` saved on Day 1, in us-east-1.

In [ ]:
session = boto3.Session()
region = session.region_name
identity = session.client("sts").get_caller_identity()
account = identity["Account"]

print(f"Working as {identity['Arn']} in {region}")

In [ ]:
lambda_client = session.client("lambda")
iam = session.client("iam")
logs = session.client("logs")
kms = session.client("kms")
s3 = session.client("s3")
secrets_manager = session.client("secretsmanager")
agentcore = session.client("bedrock-agentcore-control")
explorer = session.client("resource-explorer-2")
budgets = session.client("budgets")

## Part 1: What's there now

### UnderCover's resources

Everything we made for UnderCover is named `undercover...`, apart from what the CLI and the CDK name themselves: the stack `AgentCore-undercover-default` with the runtimes' roles, and the CDK bootstrap's `CDKToolkit` stack, with its `cdk-hnb659fds-...` roles, bucket and encryption key. As on Day 4, we find each one by name, never from a list we kept.

This looks in every service we used. For Memories it takes any whose name includes `undercover`, which covers the claim agent's and the Copilot's. The last line is the token vault, where AgentCore Identity keeps the credentials, and it says none while the vault uses AWS's own encryption key.

In [ ]:
LOG_GROUPS = ["/aws/lambda/undercover", "/aws/bedrock-agentcore/runtimes/undercover"]


def token_vault_key():
    """The key of yours that the token vault uses, if it isn't on AWS's own key."""
    key = agentcore.get_token_vault(tokenVaultId="default").get("kmsConfiguration", {})
    return [key["kmsKeyArn"]] if key.get("keyType") == "CustomerManagedKey" else []


def find_undercover():
    """Everything of UnderCover's on the account, found by name, service by service."""
    functions = [f["FunctionName"] for page in lambda_client.get_paginator("list_functions").paginate() for f in page["Functions"]]
    providers = [p["name"] for page in agentcore.get_paginator("list_api_key_credential_providers").paginate() for p in page["credentialProviders"]]
    memories = [m for page in agentcore.get_paginator("list_memories").paginate() for m in page["memories"]]
    roles = [r["RoleName"] for page in iam.get_paginator("list_roles").paginate() for r in page["Roles"]]
    log_pages = [logs.get_paginator("describe_log_groups").paginate(logGroupNamePrefix=prefix) for prefix in LOG_GROUPS]
    secrets = [s["Name"] for page in secrets_manager.get_paginator("list_secrets").paginate() for s in page["SecretList"]]
    aliases = [a["AliasName"] for page in kms.get_paginator("list_aliases").paginate() for a in page["Aliases"]]
    return {
        "Lambda functions": [name for name in functions if name.startswith("undercover")],
        "Runtimes": [r["agentRuntimeName"] for r in aws.find_runtimes("undercover")],
        "Gateways": [g["name"] for g in aws.find_gateways()],
        "Policy engines": [e["name"] for e in aws.find_policy_engines()],
        "Credential providers": [name for name in providers if name.startswith("undercover")],
        "Memories": [m["id"] for m in memories if "undercover" in m["id"] and m["status"] != "DELETING"],
        "Stacks": aws.find_stacks(),
        "IAM roles": [name for name in roles if name.startswith(("undercover", "AgentCore-undercover", "cdk-hnb659fds"))],
        "Log groups": [g["logGroupName"] for pages in log_pages for page in pages for g in page["logGroups"]],
        "Secrets": [name for name in secrets if "undercover" in name],
        "S3 buckets": [b["Name"] for b in s3.list_buckets()["Buckets"] if b["Name"].startswith("cdk-hnb659fds")],
        "Encryption keys": [name for name in aliases if name.startswith("alias/cdk-hnb659fds")],
        "Token vault key of yours": token_vault_key(),
    }


def show_undercover():
    """Print what's left of UnderCover on the account, by service."""
    for label, names in find_undercover().items():
        print(f"{label + ':':<26}{', '.join(names) or 'none'}")


show_undercover()

### Everything on your account

AWS Resource Explorer keeps an index of the resources in each Region of your account, and it's free. Since October 2025 it's on by default, and the first search in a Region sets up a full index there, which can take a while to fill. So we search now, and again at the end.

It searches the Region you work in, us-east-1, where all of the course's resources are. To search every Region, you can turn on cross-Region search in the Resource Explorer console. This counts every resource it finds by type, then lists any with `undercover` in its name. A new AWS account already has some resources, such as the default network each Region starts with, and if you use your account for other work you'll see that too.

In [ ]:
def show_everything():
    """Count every resource Resource Explorer knows of, by Region and type, and list
    any with undercover in its name."""
    try:
        pages = explorer.get_paginator("list_resources").paginate()
        resources = [r for page in pages for r in page["Resources"]]
    except explorer.exceptions.AccessDeniedException:
        print("Your IAM user isn't allowed to use Resource Explorer. The course policy from Day 1, labs/undercover-course-policy.json, allows it.")
        return
    except explorer.exceptions.UnauthorizedException:
        print("Resource Explorer isn't set up in this Region yet. Open it in the AWS console, signed in as root, then run this cell again.")
        return
    counts = Counter((r["Region"], r["ResourceType"]) for r in resources)
    for (resource_region, resource_type), count in sorted(counts.items()):
        print(f"{resource_region:<16}{resource_type:<50}{count}")
    print(f"\n{len(resources)} resources in all")
    ours = sorted(r["Arn"] for r in resources if "undercover" in r["Arn"].lower())
    print(f"\nWith undercover in the name: {len(ours)}")
    for arn in ours:
        print(f"  {arn}")


show_everything()

## Part 2: Both runtimes

As on Day 4, `agentcore remove all` empties the project's configuration, and the next deploy deletes from AWS everything the CLI deployed: the stack with both runtimes and their roles, and the three credentials in AgentCore Identity. Then we put the configuration back from git, so the repo is as it was. In your terminal, in the `undercover` folder, run:

`agentcore remove all --yes`

`agentcore deploy --yes`

`git restore agentcore/agentcore.json`

It takes about a minute. `remove all` takes the secrets out of `agentcore/.env.local`, and this takes out the four settings.

In [ ]:
LOCAL_ENV = Path("../agentcore/.env.local")
SETTINGS = ["UNDERCOVER_API", "UNDERCOVER_GATEWAY_URL", "UNDERCOVER_MEMORY_ID", "UNDERCOVER_COPILOT_MEMORY_ID"]

present = dotenv_values(LOCAL_ENV) if LOCAL_ENV.exists() else {}
for name in SETTINGS:
    if name in present:
        unset_key(LOCAL_ENV, name)
        print(f"Took {name} out of {LOCAL_ENV}")

Each runtime's log group keeps its logs forever, so we delete both by name.

In [ ]:
runtime_log_groups = [
    group["logGroupName"]
    for page in logs.get_paginator("describe_log_groups").paginate(logGroupNamePrefix="/aws/bedrock-agentcore/runtimes/undercover")
    for group in page["logGroups"]
]

for name in runtime_log_groups:
    logs.delete_log_group(logGroupName=name)
    print(f"Deleted log group {name}")

## Part 3: The Gateway, the Policy and both Memories

`aws.tear_down()` deletes Day 3's pieces, as it did on Day 4: the Gateway with its targets, the policy engine with its policies, the app's key in AgentCore Identity, the claim agent's Memory and the Gateway's role.

In [ ]:
aws.tear_down()

The Copilot's Memory is new today, and `tear_down` looks only for the claim agent's. So we delete every Memory with `undercover` in its name that isn't already being deleted. A Memory takes a few minutes to go.

In [ ]:
memories = [m for page in agentcore.get_paginator("list_memories").paginate() for m in page["memories"]]

for memory in memories:
    if "undercover" in memory["id"] and memory["status"] != "DELETING":
        agentcore.delete_memory(memoryId=memory["id"])
        print(f"Deleted memory {memory['id']}")

## Part 4: The app and the context layer

The app and the context layer have stayed up on Lambda since Days 2 and 3. Now they go too, with their roles and log groups. This deletes every Lambda function and IAM role whose name starts with `undercover`, and their log groups under `/aws/lambda/undercover`, so it also catches anything an earlier part left behind.

In [ ]:
functions = [f["FunctionName"] for page in lambda_client.get_paginator("list_functions").paginate() for f in page["Functions"]]

for name in functions:
    if name.startswith("undercover"):
        lambda_client.delete_function(FunctionName=name)
        print(f"Deleted function {name}")

for role_name in aws.find_roles():
    aws.delete_role(role_name)

function_log_groups = [
    group["logGroupName"]
    for page in logs.get_paginator("describe_log_groups").paginate(logGroupNamePrefix="/aws/lambda/undercover")
    for group in page["logGroups"]
]

for name in function_log_groups:
    logs.delete_log_group(logGroupName=name)
    print(f"Deleted log group {name}")

## Part 5: The CDK bootstrap and the token vault

### The CDK bootstrap

The CLI deploys through a CDK bootstrap: the `CDKToolkit` stack, a bucket for the deploy's files and an encryption key, which costs $1 a month. As on Day 4, we empty the bucket, delete the stack and then the bucket. Deleting the stack schedules the key's deletion, and its billing stops. It takes about a minute.

Every stack the CDK deployed needs the bootstrap to be deleted, so the bootstrap goes last. This cell first looks for stacks still deployed through it, such as UnderCover's if Part 2's commands didn't run, and deletes the bootstrap only when there are none.

In [ ]:
cloudformation = session.client("cloudformation")
stacks = [stack for page in cloudformation.get_paginator("describe_stacks").paginate() for stack in page["Stacks"]]
deployed_through_it = [s["StackName"] for s in stacks if "cdk-hnb659fds-cfn-exec-role" in s.get("RoleARN", "") and s["StackName"] != "CDKToolkit"]

if deployed_through_it:
    print(f"Keeping the bootstrap, which these stacks still need: {', '.join(deployed_through_it)}")
    print("Delete them first, then run this cell again. For an AgentCore project, that's agentcore remove all --yes and agentcore deploy --yes in its folder, as in Part 2.")
else:
    aws.delete_bootstrap()

### The token vault

AgentCore Identity keeps the credentials' secrets in a token vault, for the whole account. It uses AWS's own encryption key unless something has moved it to a key of yours, which costs $1 a month. The CLI version we use leaves it on AWS's key, but earlier versions of the CLI made a key for it, described as an "AgentCore Identity encryption key". This cell checks. If the vault is on a key the CLI made, it moves the vault back to AWS's key first, and only then schedules that key's deletion, since a key the vault still uses must never be deleted. Any other key of yours it leaves alone.

In [ ]:
CLI_KEY = "AgentCore Identity encryption key"

vault_key = agentcore.get_token_vault(tokenVaultId="default").get("kmsConfiguration", {})

if vault_key.get("keyType") != "CustomerManagedKey":
    print("The token vault uses AWS's own key, so there's nothing to delete")
elif kms.describe_key(KeyId=vault_key["kmsKeyArn"])["KeyMetadata"].get("Description", "").startswith(CLI_KEY):
    agentcore.set_token_vault_cmk(tokenVaultId="default", kmsConfiguration={"keyType": "ServiceManagedKey"})
    kms.schedule_key_deletion(KeyId=vault_key["kmsKeyArn"], PendingWindowInDays=7)
    print(f"The token vault is back on AWS's key, and {vault_key['kmsKeyArn']} will be deleted in 7 days")
else:
    print(f"The token vault uses {vault_key['kmsKeyArn']}, which the CLI didn't make, so we leave it")

## Part 6: The three checks

### UnderCover's resources

Each line should say none. A Gateway or a policy engine takes a minute to go, so if one is listed, run this cell again first. If anything else is listed, run the part that deletes it again.

In [ ]:
time.sleep(10)
show_undercover()

### Everything on your account

Resource Explorer takes a few minutes to catch up with a delete, and longer while it's still filling a new index, so what you just deleted may still be listed. Anything named `undercover` still listed should be what stays: the course's IAM user, group and policy, and `monitor.py`'s budgets, none of which costs anything. Look down the rest of the list for anything you don't recognize. Tomorrow, run the cells up to the end of Part 1 again to look once more. They only list, and delete nothing.

In [ ]:
show_everything()

### What it cost

`monitor.py` keeps two free daily budgets in AWS Budgets, one with everything your account used and one with what's left to pay after your credits. We read them the same way, for today (UTC) and the last 30 days.

In [ ]:
ALL_SPEND = "undercover-daily-spend"
AFTER_CREDITS = "undercover-daily-spend-after-credits"


def daily_spend(name):
    """Each day's spend on one of monitor.py's budgets by UTC date, and when AWS last updated it."""
    now = datetime.now(UTC)
    period = {"Start": now - timedelta(days=30), "End": now}
    pages = budgets.get_paginator("describe_budget_performance_history").paginate(AccountId=account, BudgetName=name, TimePeriod=period)
    spend = {}
    for page in pages:
        for day in page["BudgetPerformanceHistory"]["BudgetedAndActualAmountsList"]:
            if "ActualAmount" in day:
                spend[day["TimePeriod"]["Start"].astimezone(UTC).date()] = float(day["ActualAmount"]["Amount"])
    budget = budgets.describe_budget(AccountId=account, BudgetName=name)["Budget"]
    if "CalculatedSpend" in budget:
        spend[now.date()] = float(budget["CalculatedSpend"]["ActualSpend"]["Amount"])
    return spend, budget["LastUpdatedTime"]


today = datetime.now(UTC).date()
first_day = today - timedelta(days=29)

for label, name in [("Including what credits paid for", ALL_SPEND), ("What you pay after credits", AFTER_CREDITS)]:
    try:
        spend, updated = daily_spend(name)
    except budgets.exceptions.NotFoundException:
        print(f"There's no budget {name} yet. Run uv run monitor.py once, which creates it.")
        continue
    month = sum(amount for date, amount in spend.items() if date >= first_day)
    print(f"{label}: today ${spend.get(today, 0.0):,.2f}, last 30 days ${month:,.2f}")
    print(f"  AWS last updated this at {updated.astimezone():%H:%M}")

AWS updates these budgets up to three times a day, so today's figure can be several hours behind. Over the next day or two, check again with `uv run monitor.py`. For the official figures, review Billing and Cost Management in your AWS console.

## What stays

Your IAM user, the `undercover-course` group and policy, your budget with alerts from Day 1 and `monitor.py`'s two daily budgets all stay, and none of them costs anything. So do a few things AWS made for you along the way, such as the service-linked roles for AgentCore and Resource Explorer, and Resource Explorer's index. If you've finished with AWS for now, deactivate the user's access key: signed in to the console as root, open **IAM**, then **Users**, your user and **Security credentials**, and choose **Actions** and **Deactivate** on the key. You can activate it again whenever you come back.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Addition to your assignment - remove it!</h2>
            <span style="color:#EF476F;">Challenge: the solution you build for today's assignment will need a teardown too. Write one that finds everything your solution deployed by name or by tag, deletes it in an order that works, and ends with the same three checks.
            </span>
        </td>
    </tr>
</table>